In [2]:
import sqlite3
import pandas as pd

print("SQLite version:", sqlite3.sqlite_version)
print("Pandas version:", pd.__version__)

SQLite version: 3.45.3
Pandas version: 2.2.2


In [1]:
import yfinance as yf

In [3]:
%run ../src/get_data.py

Rows downloaded: 4142
First date: 2010-01-04 00:00:00
Last date: 2026-11-13 00:00:00
Saved to: C:\Users\Narendra Parab\OneDrive\Desktop\QUANTS\quant-foundations\data\nifty50.csv


In [5]:
import sqlite3
import pandas as pd
from pathlib import Path

# Connect to (or create) the database
db_path = Path("../data/market.db")
conn = sqlite3.connect(db_path)

# Read the downloaded CSV
prices = pd.read_csv("../data/nifty50.csv")

# Keep only the required columns
prices = prices[["date", "close"]].copy()

# Standardise column names
prices.columns = ["date", "close"]

# Load into SQLite
prices.to_sql(
    "prices",
    conn,
    if_exists="replace",
    index=False
)

print("Rows loaded:", len(prices))

Rows loaded: 4142


In [6]:
from pathlib import Path
import pandas as pd

sql_query = Path("../sql/01_returns.sql").read_text()

returns = pd.read_sql_query(sql_query, conn)

print("Total observations:", len(returns))
display(returns.head())

Total observations: 4142


,date,close,previous_close,daily_return
0,01-01-2020,12182.50000,NaN,NaN
1,01-01-2021,14018.50000,12182.50000,0.150708
2,01-01-2024,21741.90039,14018.50000,0.550943
3,01-01-2025,23742.90039,21741.90039,0.092034
4,01-01-2026,26146.55078,23742.90039,0.101237


In [7]:
print("Missing daily returns:", returns["daily_return"].isna().sum())

Missing daily returns: 1


In [8]:
display(pd.read_sql_query(
    "SELECT date, close FROM prices ORDER BY date LIMIT 10",
    conn
))

,date,close
0,01-01-2020,12182.500000
1,01-01-2021,14018.500000
2,01-01-2024,21741.900390
3,01-01-2025,23742.900390
4,01-01-2026,26146.550780
5,01-02-2010,4899.700195
6,01-02-2011,5417.200195
7,01-02-2012,5235.700195
8,01-02-2013,5998.899902
9,01-02-2016,7555.950195


In [9]:
display(pd.read_sql_query(
    "SELECT date, close FROM prices LIMIT 10",
    conn
))

,date,close
0,04-01-2010,5232.200195
1,05-01-2010,5277.899902
2,06-01-2010,5281.799805
3,07-01-2010,5263.100098
4,08-01-2010,5244.750000
5,11-01-2010,5249.399902
6,12-01-2010,5210.399902
7,13-01-2010,5233.950195
8,14-01-2010,5259.899902
9,15-01-2010,5252.200195


In [12]:
prices = pd.read_csv("../data/nifty50.csv")

prices["date"] = pd.to_datetime(
    prices["date"],
    dayfirst=True
)

prices = prices[["date", "close"]].copy()

prices["date"] = prices["date"].dt.strftime("%Y-%m-%d")

prices.columns = ["date", "close"]

prices.to_sql("prices", conn, if_exists="replace", index=False)

4142

In [13]:
display(pd.read_sql_query(
    "SELECT date, close FROM prices ORDER BY date LIMIT 10",
    conn
))

,date,close
0,2010-01-04,5232.200195
1,2010-01-05,5277.899902
2,2010-01-06,5281.799805
3,2010-01-07,5263.100098
4,2010-01-08,5244.750000
5,2010-01-11,5249.399902
6,2010-01-12,5210.399902
7,2010-01-13,5233.950195
8,2010-01-14,5259.899902
9,2010-01-15,5252.200195


In [14]:
returns = pd.read_sql_query(sql_query, conn)
display(returns.head())

,date,close,previous_close,daily_return
0,2010-01-04,5232.200195,NaN,NaN
1,2010-01-05,5277.899902,5232.200195,0.008734
2,2010-01-06,5281.799805,5277.899902,0.000739
3,2010-01-07,5263.100098,5281.799805,-0.003540
4,2010-01-08,5244.750000,5263.100098,-0.003487


In [15]:
print("Observations:", len(returns))
print("Missing returns:", returns["daily_return"].isna().sum())
print("Duplicate dates:", returns["date"].duplicated().sum())

Observations: 4142
Missing returns: 1
Duplicate dates: 0


In [16]:
daily_vol = returns["daily_return"].std()
annual_vol = daily_vol * (252 ** 0.5)

print("Annualised volatility:", annual_vol)

Annualised volatility: 0.16435583689456693
